In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import crosscoders as xc


------------------------- CONSTANTS -------------------------
GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/ec2-user/crosscoders',
    CONFIG_FILEPATH = '/home/ec2-user/crosscoders/src/scripts/configs/train.yml',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 8192,
        MAX_RECORDS = None,
        MAX_BATCHES = 1000000,
        MAX_TOKENS = 1000000,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)
-------------------------------------------------------------



In [3]:
from crosscoders.data.dataset import TinyStoriesRayDataset
import ray, ray.data


import os
import torch

from crosscoders.autoencoders.acausal.loss import AcausalLoss
from crosscoders.autoencoders.acausal.model import AcausalAutoencoder
from crosscoders.dataclasses.configs.runner import *
import tempfile

from crosscoders.utils import *
import torch.amp, torch.optim
import datetime, numpy as np


from torch.utils.tensorboard import SummaryWriter

In [32]:
def get_s3_keys(bucket_name, key_prefix):
    
    bucket = boto3.resource('s3').Bucket(bucket_name)

    return [
        f's3://{obj.bucket_name}/{obj.key}' 
        for obj in bucket.objects.filter(Prefix=key_prefix, Marker=key_prefix)
    ]

keys = get_s3_keys('crosscoders', 'input/roneneldan/TinyStories/train/')
len(keys)

969

In [36]:
import ray, ray.data


_ = ray.data.read_parquet(keys)
_

Metadata Fetch Progress 0:   0%|          | 0.00/161 [00:00<?, ? task/s]

Parquet Files Sample 0:   0%|          | 0.00/9.00 [00:00<?, ? file/s]

Dataset(
   num_rows=10000,
   schema={resid_post: numpy.ndarray(shape=(248, 12, 768), dtype=float)}
)

In [ ]:
_.map_batches(lambda batch: {k: np.concatenate(v) for k, v in batch.items()})

In [65]:
# for b in _.map_batches(lambda batch: {'resid_post': list(batch['resid_post'].shape)}).iter_batches(batch_size=64):
# for b in _.iter_batches(batch_size=128):
for b in _.map_batches(lambda batch: {k: np.concatenate(v) for k, v in batch.items()}).iter_batches(batch_size=128):
    print('done')
    # try:
    #     print(np.concatenate(b['resid_post']).shape)
    # except:
    #     raise
    # break


2025-02-04 15:20:50,282	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-04_14-24-40_692735_84621/logs/ray-data
2025-02-04 15:20:50,282	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> TaskPoolMapOperator[MapBatches(<lambda>)]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- MapBatches(<lambda>) 2: 0.00 row [00:00, ? row/s]

done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


2025-02-04 15:21:41,089	ERROR streaming_executor_state.py:485 -- An exception was raised from a task of operator "ReadParquet->SplitBlocks(2)". Dataset execution will now abort. To ignore this exception and continue, set DataContext.max_errored_blocks.
2025-02-04 15:21:41,103	WARNING util.py:997 -- Caught exception in filling worker!
Traceback (most recent call last):
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/util.py", line 986, in _run_filling_worker
    for idx, item in enumerate(base_iterator):
                     ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/execution/interfaces/executor.py", line 37, in __next__
    return self.get_next()
           ^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/execution/legacy_compat.py", line 76, in get_next
    bundle = self._base_iterator.get_next(output_split_idx)
   

RayTaskError(ValueError): [36mray::ReadParquet->SplitBlocks(2)()[39m (pid=122915, ip=10.0.1.126)
    for b_out in map_transformer.apply_transform(iter(blocks), ctx):
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/execution/operators/map_transformer.py", line 459, in __call__
    yield block.slice(offset, offset + size, copy=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/arrow_block.py", line 222, in slice
    view = transform_pyarrow.combine_chunks(view)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/arrow_ops/transform_pyarrow.py", line 382, in combine_chunks
    new_column_values_arrays.append(combine_chunked_array(col))
                                    ^^^^^^^^^^^^^^^^^^^^^^

In [63]:
b['resid_post']

(128,)

In [64]:
# [np.pad(_, ((0, MAX_SEQ_LEN - _.shape[0]), (0, 0), (0, 0)), EOT_VALUE) for _ in b['resid_post']]
# [_.reshape(-1, _.shape[-1]).shape for _ in b['resid_post']]
np.concatenate(b['resid_post']).shape

(32200, 12, 768)

In [51]:
import numpy as np

np.stack(b['resid_post']).shape

ValueError: all input arrays must have the same shape

In [40]:
b = _.take_batch(5)

2025-02-04 15:03:35,984	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-04_14-24-40_692735_84621/logs/ray-data
2025-02-04 15:03:35,985	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=5]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=5 2: 0.00 row [00:00, ? row/s]

AttributeError: Use `ds.count()` to compute the length of a distributed Dataset. This may be an expensive operation.

In [45]:
b['resid_post'].shape

(5, 248, 12, 768)

In [3]:



train_ds = TinyStoriesRayDataset().load('activations')
train_ds

Metadata Fetch Progress 0:   0%|          | 0.00/161 [00:00<?, ? task/s]

2025-02-04 14:32:15,732	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 10.0.1.126:6379...
2025-02-04 14:32:15,742	INFO worker.py:1841 -- Connected to Ray cluster.


Parquet Files Sample 0:   0%|          | 0.00/9.00 [00:00<?, ? file/s]

limit=1
+- Dataset(
      num_rows=10000,
      schema={resid_post: numpy.ndarray(shape=(248, 12, 768), dtype=float)}
   )

(ReadParquet->SplitBlocks(2) pid=122915) Traceback (most recent call last):
(ReadParquet->SplitBlocks(2) pid=122915)   File "pyarrow/public-api.pxi", line 145, in pyarrow.lib.pyarrow_wrap_data_type
(ReadParquet->SplitBlocks(2) pid=122915)   File "pyarrow/types.pxi", line 606, in pyarrow.lib.LargeListType.init
(ReadParquet->SplitBlocks(2) pid=122915)   File "pyarrow/types.pxi", line 232, in pyarrow.lib.DataType.init
(ReadParquet->SplitBlocks(2) pid=122915)   File "pyarrow/types.pxi", line 105, in pyarrow.lib._datatype_to_pep3118
(ReadParquet->SplitBlocks(2) pid=122915)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/air/util/tensor_extensions/arrow.py", line 461, in __arrow_ext_deserialize__
(ReadParquet->SplitBlocks(2) pid=122915)     @classmethod
(ReadParquet->SplitBlocks(2) pid=122915) 
(ReadParquet->SplitBlocks(2) pid=122915) KeyboardInterrupt: 
(ReadParquet->SplitBlocks(2) pid=122919) 
(ReadParquet->SplitBlocks(2) pid=122912) 
(ReadParquet->SplitBlocks(2)

...

(raylet) Traceback (most recent call last):
  File "python/ray/_raylet.pyx", line 1830, in ray._raylet.execute_task
  File "python/ray/_raylet.pyx", line 1864, in ray._raylet.execute_task
  File "python/ray/_raylet.pyx", line 966, in ray._raylet.raise_if_dependency_failed
ray.exceptions.RaySystemError: System error: [Errno 2] No such file or directory: '/src/scripts/configs/train.yml'
traceback: Traceback (most recent call last):
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 460, in deserialize_objects
    obj = self._deserialize_object(data, metadata, object_ref)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 317, in _deserialize_object
    return self._deserialize_msgpack_data(data, metadata_fields)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.co

(_MapWorker pid=155797) [Errno 2] No such file or directory: '/src/scripts/configs/train.yml'
(_MapWorker pid=155797) Traceback (most recent call last):
(_MapWorker pid=155797)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 460, in deserialize_objects
(_MapWorker pid=155797)     obj = self._deserialize_object(data, metadata, object_ref)
(_MapWorker pid=155797)           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(_MapWorker pid=155797)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 317, in _deserialize_object
(_MapWorker pid=155797)     return self._deserialize_msgpack_data(data, metadata_fields)
(_MapWorker pid=155797)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(_MapWorker pid=155797)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 272, in _deserialize_msgpack_data
(_MapWork

(raylet) Traceback (most recent call last):
  File "python/ray/_raylet.pyx", line 1830, in ray._raylet.execute_task
  File "python/ray/_raylet.pyx", line 1864, in ray._raylet.execute_task
  File "python/ray/_raylet.pyx", line 966, in ray._raylet.raise_if_dependency_failed
ray.exceptions.RaySystemError: System error: [Errno 2] No such file or directory: '/src/scripts/configs/train.yml'
traceback: Traceback (most recent call last):
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 460, in deserialize_objects
    obj = self._deserialize_object(data, metadata, object_ref)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 317, in _deserialize_object
    return self._deserialize_msgpack_data(data, metadata_fields)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.co

(_MapWorker pid=155911) [Errno 2] No such file or directory: '/src/scripts/configs/train.yml'
(_MapWorker pid=155911) Traceback (most recent call last):
(_MapWorker pid=155911)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 460, in deserialize_objects
(_MapWorker pid=155911)     obj = self._deserialize_object(data, metadata, object_ref)
(_MapWorker pid=155911)           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(_MapWorker pid=155911)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 317, in _deserialize_object
(_MapWorker pid=155911)     return self._deserialize_msgpack_data(data, metadata_fields)
(_MapWorker pid=155911)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(_MapWorker pid=155911)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/_private/serialization.py", line 272, in _deserialize_msgpack_data
(_MapWork

(_MapWorker pid=156036) 
(_MapWorker pid=156036) ------------------------- CONSTANTS -------------------------
(_MapWorker pid=156036) GlobalsConfig(CONFIG_FILEPATH='/home/ec2-user/crosscoders/src/scripts/configs/data.yml', PROJECT_ROOT_DIR='~/repos/crosscoders', DATA_DIR='~/repos/crosscoders/data', EXPERIMENT=ExperimentConfig(BATCH_SIZE=32, MAX_EPOCHS=1, MAX_TOKENS=None, MAX_RECORDS=10000, NUM_GPUS_ACTIVATION=0.4, NUM_GPUS=1, NUM_TRAINERS=1, HARDWARE=HardwareConfig(dtype=torch.float32, device='cuda')))
(_MapWorker pid=156036) -------------------------------------------------------------
(_MapWorker pid=156036) 
(_MapWorker pid=156036) Loaded pretrained model gpt2-small into HookedTransformer


In [8]:
b = train_ds.take_batch(5)
b

2025-02-04 14:35:11,123	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-04_14-24-40_692735_84621/logs/ray-data
2025-02-04 14:35:11,124	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=1] -> LimitOperator[limit=5]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=1 2: 0.00 row [00:00, ? row/s]

- limit=5 3: 0.00 row [00:00, ? row/s]

{'resid_post': array([[[[ 3.0908686e-01,  7.2230220e-02,  5.1841795e-01, ...,
            1.6378125e+00,  1.6544313e+00,  3.1094021e-01],
          [-6.4529341e-01, -7.7035338e-01, -1.2320495e-01, ...,
            8.8339454e-01,  6.3865554e-01, -2.8732538e-02],
          [-3.9676502e+00, -4.1807346e+00, -3.5621362e+00, ...,
           -2.5396636e+00, -2.5435274e+00, -3.2214463e+00],
          ...,
          [-5.1365967e+00, -4.9099088e+00, -4.8108511e+00, ...,
           -3.4862280e+00, -3.7754683e+00, -4.8405895e+00],
          [-5.2428398e+00, -4.7903261e+00, -5.1805501e+00, ...,
           -3.4616957e+00, -3.4614506e+00, -5.0971622e+00],
          [-7.3110306e-01,  5.6496483e-01, -1.9249140e+00, ...,
            5.5696237e-01,  1.0523720e+00, -4.5406342e-01]],
 
         [[-8.1855696e-01, -7.0431280e-01, -9.1883743e-01, ...,
           -1.1011517e+00, -6.0491943e-01,  4.4054246e-01],
          [-4.4862360e-02, -6.1156321e-01, -8.5371447e-01, ...,
           -1.2212805e+00, -1.286497

In [11]:
b['resid_post'].shape

(1, 248, 12, 768)

In [ ]:



_ = ray.data.read_parquet('s3://crosscoders/input/roneneldan/TinyStories/train/*.parquet')

FileNotFoundError: crosscoders/input/roneneldan/TinyStories/train/*.parquet

In [6]:
_.take_all()

2025-02-04 14:34:26,561	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-04_14-24-40_692735_84621/logs/ray-data
2025-02-04 14:34:26,562	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(128) 1: 0.00 row [00:00, ? row/s]

[{'resid_post': array([[[ 0.3090859 ,  0.07223251,  0.51841736, ...,  1.6378136 ,
            1.6544311 ,  0.310941  ],
          [-0.64529294, -0.7703494 , -0.12320453, ...,  0.88339627,
            0.6386531 , -0.02873352],
          [-3.967648  , -4.1807327 , -3.562141  , ..., -2.5396614 ,
           -2.5435302 , -3.2214458 ],
          ...,
          [-5.136591  , -4.909909  , -4.8108563 , ..., -3.4862244 ,
           -3.7754683 , -4.840592  ],
          [-5.2428355 , -4.7903275 , -5.1805544 , ..., -3.4616947 ,
           -3.4614503 , -5.097163  ],
          [-0.7310987 ,  0.5649644 , -1.924921  , ...,  0.5569662 ,
            1.0523756 , -0.45406622]],
  
         [[ 0.16968346, -0.03020494,  0.05502912, ..., -1.1415968 ,
            1.2892563 , -0.8234072 ],
          [ 1.1815785 ,  0.8622232 , -0.03860143, ..., -1.3034934 ,
            1.2355375 , -0.40383643],
          [ 0.03473008,  0.5015806 ,  0.48284233, ..., -1.2752504 ,
            1.065254  , -0.5127748 ],
          ...

In [8]:
train_ds = TinyStoriesRayDataset()

In [9]:
ds = train_ds.load()
ds

2025-02-04 15:32:15,101	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 10.0.1.126:6379...
2025-02-04 15:32:15,112	INFO worker.py:1841 -- Connected to Ray cluster.
2025-02-04 15:32:15,166	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-04_14-24-40_692735_84621/logs/ray-data
2025-02-04 15:32:15,166	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadHuggingFace]


Running 0: 0.00 row [00:00, ? row/s]

- ReadHuggingFace->SplitBlocks(14) 1: 0.00 row [00:00, ? row/s]

MapBatches(TokenToLatents)
+- limit=10000
   +- Dataset(num_rows=?, schema={text: object})

In [12]:
b = ds.take_batch(5)
b

2025-02-04 15:33:34,057	WARNING map_operator.py:701 -- Specifying both num_cpus and num_gpus for map tasks is experimental, and may result in scheduling or stability issues. Please report any issues to the Ray team: https://github.com/ray-project/ray/issues/new/choose
2025-02-04 15:33:34,060	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-04_14-24-40_692735_84621/logs/ray-data
2025-02-04 15:33:34,060	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadHuggingFace] -> LimitOperator[limit=10000] -> ActorPoolMapOperator[MapBatches(TokenToLatents)] -> LimitOperator[limit=5]


Running 0: 0.00 row [00:00, ? row/s]

KeyboardInterrupt: 

In [4]:
import yaml


with open('/home/ec2-user/crosscoders/src/scripts/configs/train.yml', 'r') as infl:
    cfg = yaml.safe_load(infl)

cfg

{'GLOBALS': {'PROJECT_ROOT_DIR': '~/repos/crosscoders',
  'CONFIG_FILEPATH': None,
  'DATA_DIR': '~/repos/crosscoders/data',
  'EXPERIMENT': {'BATCH_SIZE': 50000, 'MAX_EPOCHS': 1}},
 'RUNNER': {'MODEL': {'CAUSALITY': 'acausal'}}}

In [5]:
from crosscoders.configs import *

In [6]:
from crosscoders.utils import from_dict, dataclass_repr

In [7]:
cfg = from_dict(RunnerConfig, cfg.get('RUNNER', {}))
cfg

AutoencoderLightningModuleConfig(
    OPTIMIZER = OptimizerConfig(
        optimizer = <class 'torch.optim.adam.Adam'>,
        parameters = OptimizerParameters(
            lr = 0.001,
            betas = (0.9, 0.999),
        ),
    ),
    MODEL = ModelConfig(
        CAUSALITY = 'acausal',
        LOCALITY = 'global',
        N_LAYERS = 12,
        D_MODEL = 768,
        D_CODER = 2048,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)

In [8]:
print(repr(cfg))

AutoencoderLightningModuleConfig(OPTIMIZER=OptimizerConfig(optimizer=<class 'torch.optim.adam.Adam'>, parameters=OptimizerParameters(lr=0.001, betas=(0.9, 0.999))), MODEL=ModelConfig(CAUSALITY='acausal', LOCALITY='global', N_LAYERS=12, D_MODEL=768, D_CODER=2048, HARDWARE=HardwareConfig(dtype=torch.float32, device='cuda')))


In [17]:
type(repr(cfg))

str

In [8]:
from crosscoders.utils import *

In [14]:
print(pretty_repr(cfg))

AutoencoderLightningModuleConfig(
    OPTIMIZER = OptimizerConfig(
        optimizer = <class 'torch.optim.adam.Adam'>,
        parameters = OptimizerParameters(lr=0.001, betas=(0.9, 0.999)),
    ),
    MODEL = ModelConfig(
        CAUSALITY = 'acausal',
        LOCALITY = 'global',
        N_LAYERS = 12,
        D_MODEL = 768,
        D_CODER = 2048,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)


In [1]:
import ray.train.lightning

In [8]:


from crosscoders.configs.runner import RunnerConfig
from crosscoders.autoencoders.acausal.runner import AcausalAutoencoderRunner
from crosscoders.constants import CONSTANTS
from crosscoders.utils import *



In [9]:

model = AcausalAutoencoderRunner(
    from_dict(
        RunnerConfig, 
        get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
    )
)

In [12]:
model.model.W_dec.is_contiguous()

False

In [9]:
# import ray, ray.train
from crosscoders import CONSTANTS

CONSTANTS
print(CONSTANTS)

GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/ec2-user/crosscoders',
    CONFIG_FILEPATH = '/home/ec2-user/crosscoders/src/scripts/configs/train.yml',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 8192,
        MAX_RECORDS = None,
        MAX_BATCHES = 10,
        MAX_TOKENS = 81920,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)


In [16]:
CONSTANTS.EXPERIMENT.BATCH_SIZE = 30000
CONSTANTS.EXPERIMENT.MAX_BATCHES = (10000000 // CONSTANTS.EXPERIMENT.BATCH_SIZE) + 1
CONSTANTS.EXPERIMENT.MAX_TOKENS = 1000000

In [12]:
# # CONSTANTS.EXPERIMENT.BATCH_SIZE = 8192
# # CONSTANTS.EXPERIMENT.MAX_RECORDS = 8192 * 50
# # CONSTANTS.EXPERIMENT.MAX_RECORDS = 8192 * 20

# # CONSTANTS.EXPERIMENT.BATCH_SIZE = int(16384 * 2)
# CONSTANTS.EXPERIMENT.BATCH_SIZE = 8192
# CONSTANTS.EXPERIMENT.MAX_RECORDS = CONSTANTS.EXPERIMENT.BATCH_SIZE * 10
# # CONSTANTS.EXPERIMENT.MAX_RECORDS = 
# CONSTANTS.EXPERIMENT.MAX_BATCHES = 10
# CONSTANTS.EXPERIMENT.MAX_TOKENS = CONSTANTS.EXPERIMENT.BATCH_SIZE * CONSTANTS.EXPERIMENT.MAX_BATCHES
# # CONSTANTS.EXPERIMENT.MAX_UPDATES = 1000000

# # CONSTANTS.EXPERIMENT.MAX_EPOCHS = 2

AttributeError: 'NoneType' object has no attribute 'EXPERIMENT'

In [17]:
from crosscoders import CONSTANTS

CONSTANTS

GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/ec2-user/crosscoders',
    CONFIG_FILEPATH = '/home/ec2-user/crosscoders/src/scripts/configs/train.yml',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 30000,
        MAX_RECORDS = None,
        MAX_BATCHES = 334,
        MAX_TOKENS = 1000000,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)

In [18]:
train_ds = TinyStoriesRayDataset().load('activations')

    
# train_dl = ray.train.get_dataset_shard('train').iter_torch_batches(
train_dl = train_ds.iter_torch_batches(
    # prefetch_batches=10,
    batch_size=CONSTANTS.EXPERIMENT.BATCH_SIZE,
    # collate_fn=lambda _: {k: torch.as_tensor(np.stack(v)) for k, v in _.items()},
    # collate_fn=lambda _: {k: torch.as_tensor(np.stack([np.pad(_, ()) for _ in v])) for k, v in _.items()},
    # local_shuffle_buffer_size=16
    device='cuda'
)

Metadata Fetch Progress 0:   0%|          | 0.00/437 [00:00<?, ? task/s]

Parquet Files Sample 0:   0%|          | 0.00/10.0 [00:00<?, ? file/s]

In [15]:
# def get_stats(batch):

#     return {
#         'min': np.array([batch['resid_post'].min()]),
#         'max': np.array([batch['resid_post'].max()])
#     }
#     # return {
#     #     'min': batch['resid_post'].min(),
#     #     'max': batch['resid_post'].max()
#     # }

# ds = train_ds.map_batches(get_stats, batch_size=10000)
# # ds.cache()


# X_MIN = ds.select_columns('min').min()
# X_MAX = ds.select_columns('max').max()

2025-02-07 22:58:09,886	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-06_23-39-24_723500_3123/logs/ray-data
2025-02-07 22:58:09,886	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=300000] -> TaskPoolMapOperator[MapBatches(get_stats)->Project] -> LimitOperator[limit=1]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=300000 2: 0.00 row [00:00, ? row/s]

- MapBatches(get_stats)->Project 3: 0.00 row [00:00, ? row/s]

...

- Aggregate 4: 0.00 row [00:00, ? row/s]

Sort Sample 5:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

Shuffle Map 6:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

Shuffle Reduce 7:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

- limit=1 8: 0.00 row [00:00, ? row/s]

In [20]:
# Add this before and after each optimizer step
def print_memory_stats():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f}GB")
    
    # Check optimizer state sizes
    total_state_size = 0
    for group in optimizer.state.values():
        for state in group.values():
            if torch.is_tensor(state):
                total_state_size += state.numel() * state.element_size()
    print(f"Optimizer state size: {total_state_size / 1e9:.2f}GB")
    
    # Check if any gradients are None
    none_grads = sum(1 for p in model.parameters() if p.grad is None)
    print(f"Parameters with None gradients: {none_grads}")

In [10]:

# import os
# import torch

# from crosscoders.autoencoders.acausal.loss import AcausalLoss
# from crosscoders.autoencoders.acausal.model import AcausalAutoencoder
# from crosscoders.dataclasses.configs.runner import *
# import tempfile

# from crosscoders.utils import *
# import torch.amp, torch.optim
# import datetime, numpy as np




# cfg = from_dict(
#     RunnerConfig,
#     get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
# )

# model = AcausalAutoencoder(cfg.MODEL)
# model.to('cuda')

# criterion = AcausalLoss()

# # optimizer = cfg.OPTIMIZER.optimizer(
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     **cfg.OPTIMIZER.parameters.asdict()
# )

# scaler = torch.amp.GradScaler("cuda", enabled=False)


# for epoch in range(CONSTANTS.EXPERIMENT.NUM_EPOCHS):

# # with torch.profiler.profile(
# #         schedule=torch.profiler.schedule(wait=5, warmup=10, active=25, repeat=0),
# #         # schedule=torch.profiler.schedule(wait=5, warmup=10, active=25, repeat=1),
# #         # schedule=torch.profiler.schedule(wait=1, warmup=3, active=5, repeat=1),
# #         on_trace_ready=torch.profiler.tensorboard_trace_handler(f'./log/{datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d_%H:%M:%S")}'),
# #         record_shapes=True,
# #         profile_memory=True,
# #         # with_stack=True,
# #         # with_modules=True,

# # ) as prof:

#     model.train()
#     for batch_idx, batch in enumerate(train_dl):
#         # prof.step()
#         # This is done by `prepare_data_loader`!
#         # images, labels = images.to("cuda"), labels.to("cuda")
#         with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=False):
#             outputs = model(batch['resid_post'])
#             loss = criterion(outputs, batch['resid_post'], W_dec=model.W_dec, x_enc=model.x_enc)

#         scaler.scale(loss.loss).backward()
#         grad_norms = [param.grad.norm().item() for param in model.parameters() if param.grad is not None]
#         print(np.mean(grad_norms), np.std(grad_norms), np.min(grad_norms), np.max(grad_norms))

#         scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 100)
#         # print_memory_stats()
#         scaler.step(optimizer)
#         # print_memory_stats()
#         scaler.update()

#         # loss.loss.backward()
#         # optimizer.step()
#         optimizer.zero_grad()


#         if batch_idx % 1 == 0:
#             metrics = {
#                 'loss': loss.loss.item(),
#                 'error': loss.error.item(),
#                 'l1': loss.l1.item(),
#                 'l0': loss.l0.item(),
#             }
#             print(batch_idx, metrics)

2025-02-07 21:13:42,640	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-06_23-39-24_723500_3123/logs/ray-data
2025-02-07 21:13:42,641	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=409600]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=409600 2: 0.00 row [00:00, ? row/s]

15873.9182305336 15382.996175206401 31.786813735961914 33183.5
0 {'loss': 693262.25, 'error': 693035.375, 'l1': 226.88494873046875, 'l0': 8146.18359375}
48055.6077003479 52803.047324385334 89.29725646972656 128201.4609375
1 {'loss': 683634.4375, 'error': 682271.75, 'l1': 1362.7054443359375, 'l0': 12436.03515625}
68118.10859394073 97115.01265224296 43.49553298950195 234332.203125
2 {'loss': 662834.0, 'error': 660044.875, 'l1': 2789.10888671875, 'l0': 13376.1484375}
80334.4649477005 114717.40704595389 50.51103591918945 276677.28125
3 {'loss': 606251.375, 'error': 602502.375, 'l1': 3749.0048828125, 'l0': 13803.65234375}
103964.22037887573 135364.76357885037 100.49351501464844 330877.78125
4 {'loss': 590873.875, 'error': 586145.75, 'l1': 4728.1240234375, 'l0': 13961.9853515625}
114376.77529525757 153705.34892566525 84.79014587402344 374239.21875
5 {'loss': 544557.375, 'error': 538753.25, 'l1': 5804.10546875, 'l0': 14219.31640625}
121496.0765838623 165779.13832349653 65.901611328125 402728.

In [7]:
%load_ext dotenv

In [8]:
%dotenv -o ./.env

In [54]:
def get_sae_explained_variance(original, reconstructed):
    total_variance = np.var(original, axis=0)
    reconstruction_error = np.mean((original - reconstructed)**2, axis=0)
    explained = 1 - (reconstruction_error / total_variance)
    return explained

In [59]:
NUM_BATCHES = CONSTANTS.EXPERIMENT.MAX_RECORDS / CONSTANTS.EXPERIMENT.BATCH_SIZE

In [60]:
# cfg = from_dict(
#     RunnerConfig,
#     get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
# )

# model = AcausalAutoencoder(cfg.MODEL)
# model.to('cuda')

# criterion = AcausalLoss()

# # optimizer = cfg.OPTIMIZER.optimizer(
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     **cfg.OPTIMIZER.parameters.asdict()
# )

# scaler = torch.amp.GradScaler("cuda", enabled=False)

# writer = SummaryWriter(f'./log/{datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d_%H:%M:%S")}')
# num_tokens_processed = 0

# for epoch in range(CONSTANTS.EXPERIMENT.MAX_EPOCHS):


#     model.train()
#     for batch_idx, batch in enumerate(train_dl):
#         # prof.step()
#         # This is done by `prepare_data_loader`!
#         # images, labels = images.to("cuda"), labels.to("cuda")
#         with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=False):
#             # x = (batch['resid_post'] - X_MIN) / (X_MAX - X_MIN)
#             x = batch['resid_post']
            
#             outputs = model(x)
#             loss = criterion(outputs, x, W_dec=model.W_dec, x_enc=model.x_enc)

#             num_tokens_processed += x.shape[0]

#         scaler.scale(loss.loss).backward()
#         scaler.unscale_(optimizer)
        
        
#         grad_norms = {name: param.grad.norm().item() for name, param in model.named_parameters() if param.grad is not None}
#         weight_norms = [v for k, v in grad_norms.items() if k in ('W_enc', 'W_dec')]
#         # print(np.mean(grad_norms), np.std(grad_norms), np.min(grad_norms), np.max(grad_norms))

#         writer.add_scalar(f'W_grad_norm', np.mean(weight_norms), num_tokens_processed)
#         # writer.add_scalar(f'epoch_{epoch}/W_grad_norm/std', np.std(weight_norms), batch_idx)
#         # writer.add_scalar(f'epoch_{epoch}/W_grad_norm/min', np.min(weight_norms), batch_idx)
#         # writer.add_scalar(f'epoch_{epoch}/W_grad_norm/max', np.max(weight_norms), batch_idx)
#         # torch.nn.utils.clip_grad_norm_(model.parameters(), 100)
#         # total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
#         total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
#         writer.add_scalar(f'total_grad_norm', total_grad_norm, num_tokens_processed)

#         scaler.step(optimizer)
#         scaler.update()


#         optimizer.zero_grad()

#         # writer.add_scalars(f'Loss/loss', {
#         #     f'epoch_{epoch}': loss.loss.item()
#         # }, batch_idx)

#         writer.add_scalar(f'batch/loss', loss.loss.item(), num_tokens_processed)
#         writer.add_scalar(f'batch/error', loss.error.item(), num_tokens_processed)
#         writer.add_scalar(f'batch/l1', loss.l1.item(), num_tokens_processed)
#         writer.add_scalar(f'batch/l0', loss.l0.item(), num_tokens_processed)
#         # writer.add_scalar(f'batch/l0', loss.l0.item(), batch_idx + (epoch * NUM_BATCHES))
#         # _ = get_sae_explained_variance(x.cpu().numpy(), outputs.detach().cpu().numpy())
#         # writer.add_scalar(f'epoch_{epoch}/expl_var', _, batch_idx)


#         # if batch_idx % 1 == 0:
#         #     metrics = {
#         #         'loss': loss.loss.item(),
#         #         'error': loss.error.item(),
#         #         'l1': loss.l1.item(),
#         #         'l0': loss.l0.item(),
#         #     }
#         #     print(batch_idx, metrics)

# writer.close()

2025-02-08 00:01:51,799	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-06_23-39-24_723500_3123/logs/ray-data
2025-02-08 00:01:51,800	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=1500000]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=1500000 2: 0.00 row [00:00, ? row/s]

2025-02-08 00:04:42,342	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-06_23-39-24_723500_3123/logs/ray-data
2025-02-08 00:04:42,342	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=1500000]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=1500000 2: 0.00 row [00:00, ? row/s]

In [ ]:
import einops

feature_decoder_norms = einops.reduce(
    torch.norm(model.W_dec, dim=-1),
    'd_coder n_layers -> d_coder',
    ''
)
l1 = einops.einsum(
    model.x_enc, feature_decoder_norms,
    '... d_coder , d_coder -> ...'
).mean()

In [27]:
cfg = from_dict(
    RunnerConfig,
    get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
)

model = AcausalAutoencoder(cfg.MODEL)
model.to('cuda')

criterion = AcausalLoss()

# optimizer = cfg.OPTIMIZER.optimizer(
optimizer = torch.optim.AdamW(
    model.parameters(),
    **cfg.OPTIMIZER.parameters.asdict()
)

scaler = torch.amp.GradScaler("cuda", enabled=False)

writer = SummaryWriter(f'./log/{datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d_%H:%M:%S")}')
num_tokens_processed = 0





model.train()
for batch_idx, batch in enumerate(train_dl):

    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=False):
        x = batch['resid_post']

        outputs = model(x)
        loss = criterion(outputs, x, W_dec=model.W_dec, x_enc=model.x_enc)

        num_tokens_processed += x.shape[0]

    scaler.scale(loss.loss).backward()
    scaler.unscale_(optimizer)
    
    
    # grad_norms = {name: param.grad.norm().item() for name, param in model.named_parameters() if param.grad is not None}
    # weight_norms = [v for k, v in grad_norms.items() if k in ('W_enc', 'W_dec')]
    # # print(np.mean(grad_norms), np.std(grad_norms), np.min(grad_norms), np.max(grad_norms))

    # writer.add_scalar(f'W_grad_norm', np.mean(weight_norms), num_tokens_processed)
    # writer.add_scalar(f'epoch_{epoch}/W_grad_norm/std', np.std(weight_norms), batch_idx)
    # writer.add_scalar(f'epoch_{epoch}/W_grad_norm/min', np.min(weight_norms), batch_idx)
    # writer.add_scalar(f'epoch_{epoch}/W_grad_norm/max', np.max(weight_norms), batch_idx)
    # torch.nn.utils.clip_grad_norm_(model.parameters(), 100)
    # total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
    total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
    writer.add_scalar(f'total_grad_norm', total_grad_norm, num_tokens_processed)

    scaler.step(optimizer)
    scaler.update()


    optimizer.zero_grad()

    # writer.add_scalars(f'Loss/loss', {
    #     f'epoch_{epoch}': loss.loss.item()
    # }, batch_idx)

    writer.add_scalar(f'loss/loss', loss.loss.item(), num_tokens_processed)
    writer.add_scalar(f'loss/error', loss.error.item(), num_tokens_processed)
    writer.add_scalar(f'loss/l1', loss.l1.item(), num_tokens_processed)
    writer.add_scalar(f'loss/l0', loss.l0.item(), num_tokens_processed)
    # writer.add_scalar(f'batch/l0', loss.l0.item(), batch_idx + (epoch * NUM_BATCHES))
    # _ = get_sae_explained_variance(x.cpu().numpy(), outputs.detach().cpu().numpy())
    # writer.add_scalar(f'epoch_{epoch}/expl_var', _, batch_idx)


    # if batch_idx % 1 == 0:
    #     metrics = {
    #         'loss': loss.loss.item(),
    #         'error': loss.error.item(),
    #         'l1': loss.l1.item(),
    #         'l0': loss.l0.item(),
    #     }
    #     print(batch_idx, metrics)


writer.close()

2025-02-10 20:07:55,887	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-10_18-15-02_212237_3248/logs/ray-data
2025-02-10 20:07:55,887	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=1000000]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=1000000 2: 0.00 row [00:00, ? row/s]

In [26]:
model.W_dec.norm(dim=-1).sum(-1).max()

tensor(0.4123, device='cuda:0', grad_fn=<MaxBackward1>)

In [56]:
_

array([[-3.8902879e-01, -9.8752975e-04, -6.3213110e-03, ...,
        -1.4964223e-01, -6.8521261e-02, -1.5882170e-01],
       [-3.7786376e-01, -8.5590601e-02,  1.3673306e-03, ...,
        -4.1912711e-01, -2.0174742e-02, -1.3744199e-01],
       [-1.7545152e-01, -2.1981716e-02, -1.4014959e-02, ...,
        -2.7307928e-01, -2.4434328e-03, -1.2377238e-01],
       ...,
       [ 4.4411421e-04, -1.5540588e-01, -7.6513886e-02, ...,
        -1.0780215e-02,  2.4758577e-03, -9.3395710e-03],
       [ 2.6327372e-04, -1.3994730e-01, -1.5146554e-01, ...,
        -3.0630827e-03, -4.2017698e-03, -3.3605337e-02],
       [-3.1684160e-02, -1.5997648e-02, -1.3630331e-01, ...,
        -1.6712189e-02, -5.0508380e-02, -1.6495466e-02]], dtype=float32)

In [4]:
from ray.experimental.tqdm_ray import tqdm
import time




for epoch_idx in tqdm(range(CONSTANTS.EXPERIMENT.MAX_EPOCHS), desc='Epoch', position=0):
    batch_pbar = tqdm(range(10), desc='Batch', position=1)

    for batch_idx in batch_pbar:
        time.sleep(0.2)
    #     batch_pbar.update()
    # batch_pbar.reset()
    # epoch_pbar.update()


# epoch_pbar.close()
# batch_pbar.close()

    

NameError: name 'CONSTANTS' is not defined

In [4]:
import einops


# torch.matmul(model.x_enc, model.W_dec.norm(dim=-1).sum(-1))
(model.x_enc @ model.W_dec.norm(dim=-1).sum(-1)).mean()

NameError: name 'model' is not defined

In [75]:

feature_decoder_norms = einops.reduce(
    torch.norm(model.W_dec, dim=-1),
    'd_coder n_layers -> d_coder',
    'sum'
)
l1 = einops.einsum(
    model.x_enc, feature_decoder_norms,
    '... d_coder , d_coder -> ...'
).mean()

In [76]:
l1

tensor(4040.6978, device='cuda:0', grad_fn=<MeanBackward0>)

In [ ]:
# x = (batch['resid_post'] - X_MIN) / (X_MAX - X_MIN)

# outputs = model(x)
# loss = criterion(outputs, x, W_dec=model.W_dec, x_enc=model.x_enc)


tensor(0., device='cuda:0', grad_fn=<SumBackward0>)